## Cleaned Data Load Karna Aur Time Features Extract Karna

In [4]:
# Import Libraries & Load Cleaned Data
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# NLTK ke zaroori packages download kar rahe hain
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vishw\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vishw\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
# Data load karein
df = pd.read_csv('../data/cleaned_online_retail.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [ ]:
# Sirf Date nikal rahe hain (Time ko hata kar) taaki daily analysis kar sakein
df['Date'] = df['InvoiceDate'].dt.date
print("Data loaded successfully! Total records:", df.shape[0])

Data loaded successfully! Total records: 805620


In [10]:
# Cell 2: Creating Time-Based Features
df['Date'] = pd.to_datetime(df['Date']) # Ensure datetime format

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek

# Weekend feature: Agar Saturday(5) ya Sunday(6) hai toh 1, nahi toh 0
df['Is_Weekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

print("Time features created successfully!")
df[['Date', 'Month', 'DayOfWeek', 'Is_Weekend']].head()

Time features created successfully!


,Date,Month,DayOfWeek,Is_Weekend
0,2009-12-01,12,1,0
1,2009-12-01,12,1,0
2,2009-12-01,12,1,0
3,2009-12-01,12,1,0
4,2009-12-01,12,1,0


In [11]:
# Cell 3: Grouping Data by Date and Product Description
# Hum Quantity aur Total_Sales ko sum karenge, aur baaki time features ka mean/first le lenge
daily_demand = df.groupby(['Date', 'Description']).agg({
    'Quantity': 'sum',
    'Total_Sales': 'sum',
    'Month': 'first',
    'DayOfWeek': 'first',
    'Is_Weekend': 'first'
}).reset_index()

print("Data aggregated to Daily Demand level!")
print("New Shape (Product-Daily level):", daily_demand.shape)
display(daily_demand.head())

Data aggregated to Daily Demand level!
New Shape (Product-Daily level): (440446, 7)


,Date,Description,Quantity,Total_Sales,Month,DayOfWeek,Is_Weekend
0,2009-12-01,3 STRIPEY MICE FELTCRAFT,5,9.75,12,1,0
1,2009-12-01,BLACK PIRATE TREASURE CHEST,6,9.90,12,1,0
2,2009-12-01,BROWN PIRATE TREASURE CHEST,12,19.80,12,1,0
3,2009-12-01,CAMPHOR WOOD PORTOBELLO MUSHROOM,1,1.25,12,1,0
4,2009-12-01,PEACE WOODEN BLOCK LETTERS,3,20.85,12,1,0


In [12]:
# Cell 4: NLP Feature Extraction
stop_words = set(stopwords.words('english'))

def extract_nlp_features(text):
    if pd.isna(text):
        return 0
    
    # Tokenization and lowercase
    tokens = word_tokenize(str(text).lower())
    
    # Khass seasonal keywords jo demand impact karte hain
    seasonal_keywords = {'christmas', 'vintage', 'heart', 'love', 'pink', 'blue', 'bag', 'box', 'set'}
    
    # Check agar koi keyword description mein hai
    for token in tokens:
        if token in seasonal_keywords:
            return 1 # High seasonal probability
    return 0

# Ek naya feature bana rahe hain: 'Is_Seasonal_Product'
daily_demand['Is_Seasonal_Product'] = daily_demand['Description'].apply(extract_nlp_features)

print("NLP Sentiment/Seasonal features integrated successfully!")
print("Seasonal products count:", daily_demand['Is_Seasonal_Product'].sum())

NLP Sentiment/Seasonal features integrated successfully!
Seasonal products count: 174155


In [13]:
# Cell 5: Save Feature Engineered Dataset
daily_demand.to_csv('../data/featured_daily_demand.csv', index=False)
print("Master Feature Dataset saved as 'featured_daily_demand.csv' successfully! 🎉")

Master Feature Dataset saved as 'featured_daily_demand.csv' successfully! 🎉
